# EconoCausal — Week 1: Causal Directed Acyclic Graph (DAG) & Identification with DoWhy
## Personalized Retail Pricing and Discount Optimization via Causal Machine Learning

---

### 1. Project Context & Objectives
**Project:** EconoCausal — Dynamic Pricing via Double Machine Learning  
**Domain:** Computational Economics, Causal AI, Personalized Retail Analytics  
**Current Phase:** **Week 1 — Causal Graphing and Identification**

In retail and e-commerce, promotional discounts are among the largest operating expenditures. Standard predictive machine learning models optimize for **correlation** rather than **causation**: they predict $P(\text{Purchase} \mid \text{Discount})$, which is plagued by **selection bias** and **confounding**. For example, when marketing algorithms target historical churn-risk customers with large discounts, a naive regression can misleadingly estimate that discounts *reduce* purchasing, simply because churn-risk customers have intrinsically lower baseline conversion rates. Conversely, offering discounts to loyal customers subsidizes sales that would have happened anyway (deadweight margin loss).

Causal AI solves this fundamental policy problem by answering **counterfactual questions**:
> *"What would this specific customer's purchase probability be if we offered a $15\%$ discount versus no discount at all ($do(\text{Discount} = 0.15)$ vs. $do(\text{Discount} = 0.0)$)?"*

---

### 2. Week 1 Scope & Central Research Question
In this notebook, we focus strictly on **Week 1 foundational causal modeling**:

$$\mathbf{\text{Central Research Question:}}$$
> **How does the assigned discount percentage affect the probability that a customer purchases, after accounting for relevant pre-treatment customer characteristics?**

#### Key Deliverables for Week 1:
1. **Synthetic Retail Data Generation:** Create a realistic, reproducible dataset of 5,000 customers with observable pre-treatment characteristics, continuous discount treatment ($0\%$ to $30\%$), confounded treatment assignment, and heterogeneous treatment effects.
2. **Feature Causal Taxonomy:** Document every feature's business meaning, generation mechanism, and structural causal role (Identifier, Confounder, Treatment, Primary Outcome, Downstream Business Metric).
3. **Data Quality & Distribution Audits:** Run programmatic assertions and visualize confounded treatment assignment and conversion distributions.
4. **Causal Directed Acyclic Graph (DAG):** Formulate the causal assumptions in DOT format, document every directed edge, and verify graph acyclicity.
5. **Causal Identification with Microsoft DoWhy:** Initialize DoWhy's `CausalModel`, identify the non-parametric causal estimand via the **Backdoor Criterion**, and formalize all structural assumptions.
6. **Robust DAG Visualization:** Render an elegant, publication-quality causal graph with NetworkX and Matplotlib, ensuring compatibility and avoiding known `get_strict()` version bugs.

---

> **Strict Scope Boundary (Week 1 Only):**  
> We deliberately do **not** train Double Machine Learning (DML) models, estimate CATE/ITE, plot Qini/uplift curves, run DoWhy refutation tests, perform SciPy discount optimization, or build Streamlit dashboards in this notebook. Those belong to subsequent weeks. Week 1 establishes the sound causal foundation upon which all future estimation depends.


## 2. Dependency Installation and Imports

To ensure compatibility across operating systems (Windows, macOS, Linux) and avoid common version conflicts between `networkx` and `pydot` (such as the legacy `Graph.get_strict() takes 1 positional argument but 2 were given` error), this project uses modern, mutually compatible package specifications:

- `dowhy>=0.11`
- `pandas>=2.0`
- `numpy>=1.24`
- `matplotlib>=3.7`
- `networkx>=3.2`
- `pydot>=3.0`
- `scipy>=1.10`

Run the cell below in your Jupyter environment. If packages are newly installed or upgraded, restart your Jupyter kernel before proceeding.


In [1]:
# Step 2.1: Optional pip install (uncomment if running in a fresh environment)
%pip install -q dowhy pandas numpy matplotlib networkx pydot scipy

# Step 2.2: Standard library and third-party imports
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import networkx as nx
from dowhy import CausalModel

# Safe fallback for IPython.display
try:
    from IPython.display import display
except ImportError:
    display = print

# Step 2.3: Set global random seed for strict scientific reproducibility
RANDOM_SEED = 42
rng = np.random.default_rng(RANDOM_SEED)

# Step 2.4: Display preferences
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 1000)
pd.set_option("display.precision", 4)
plt.rcParams["figure.dpi"] = 100
plt.rcParams["font.sans-serif"] = "DejaVu Sans"

print("Libraries imported successfully!")
print(f"Random Seed set to: {RANDOM_SEED}")


Note: you may need to restart the kernel to use updated packages.
Libraries imported successfully!
Random Seed set to: 42


## 3. Synthetic Retail Data Generation

### 3.1 Data-Generating Process (DGP) Design
To evaluate causal methods rigorously, we need ground-truth causal mechanisms where confounding and effect heterogeneity are explicitly known and controlled.

We simulate $N = 5,000$ retail customers under the following structural timeline:
1. **Pre-treatment customer characteristics:**
   - `tenure_days`: How long the customer has had an account with the retailer (30 to 1,500 days).
   - `previous_orders`: Number of prior purchases, positively driven by tenure: $\text{Poisson}(\lambda = \text{tenure}/90)$.
   - `days_since_last_order`: Recency of last purchase (1 to 180 days), simulated via a Gamma distribution with an inactivity penalty for infrequent buyers.
   - `avg_order_value`: Historical average basket size ($12 to $250), positively correlated with order frequency.
   - `prior_email_engagement`: Historical email marketing engagement score $\in [0, 1]$ from a $\text{Beta}(2, 5)$ distribution.
   - `price_sensitivity`: Observable customer psychographic responsiveness to price changes $\in [0, 1]$ from a $\text{Beta}(3, 3)$ distribution.
2. **Confounded Treatment Assignment (`discount_percent`):**
   - In modern retail pricing, discounts are **not** assigned randomly. Algorithmic pricing targets:
     - Churn risks (higher `days_since_last_order`) with reactivation offers.
     - Price-sensitive shoppers (higher `price_sensitivity`) with targeted incentives.
     - Email-engaged customers (higher `prior_email_engagement`) who open promotional campaigns.
     - High-spending loyalists (higher `avg_order_value`) with *lower* discounts to prevent margin cannibalization.
   - We simulate a continuous discount range $[0.0, 0.30]$ ($0\%$ to $30\%$) with a zero-hurdle: approximately $35\%$ of customers receive no discount ($0\%$), while the remainder receive positive personalized discounts.
3. **Primary Outcome Generation (`purchased`):**
   - Follow-up purchase conversion ($0$ or $1$) is generated via a logistic model combining customer baseline characteristics and the assigned discount.
   - **Treatment Effect Heterogeneity:** The true individual treatment effect (CATE) depends on `price_sensitivity`:
     $$\tau_i = 1.5 + 4.5 \cdot \text{price\_sensitivity}_i$$
     Customers with high price sensitivity experience a substantial boost in conversion probability from discounts, while price-insensitive customers respond only modestly.
4. **Downstream Business Metric (`revenue`):**
   - If `purchased == 0`, `revenue = $0.00`.
   - If `purchased == 1`, `revenue = avg_order_value * (1 - discount_percent) * basket_multiplier`, capturing post-discount cashflow.
5. **Separation of Counterfactuals:**
   - True counterfactual parameters and unobserved potential outcomes are kept strictly internal to the generator and **never** included in the observed dataset `retail_df`.


In [2]:
def generate_synthetic_retail_data(n_customers: int = 5000, seed: int = 42) -> pd.DataFrame:
    """
    Generates a reproducible synthetic retail dataset with observable confounding
    and heterogeneous treatment effects for causal pricing optimization.
    
    Parameters:
    -----------
    n_customers : int
        Number of customer records to simulate (default: 5000).
    seed : int
        Random seed for NumPy random generator (default: 42).
        
    Returns:
    --------
    pd.DataFrame
        Observed customer dataset with 10 columns.
    """
    local_rng = np.random.default_rng(seed)
    
    # -------------------------------------------------------------
    # 1. Pre-treatment customer characteristics (Exogenous / Historical)
    # -------------------------------------------------------------
    # Customer unique identifier
    customer_id = [f"CUST_{i:05d}" for i in range(1, n_customers + 1)]
    
    # Tenure with retailer in days (30 days to ~4 years)
    tenure_days = local_rng.integers(30, 1500, size=n_customers)
    
    # Cumulative orders prior to campaign (Poisson driven by tenure)
    poisson_lam = np.clip(tenure_days / 90.0, 1.0, 15.0)
    previous_orders = local_rng.poisson(lam=poisson_lam, size=n_customers).clip(0, 50)
    
    # Days since last order (Recency, Gamma distributed; higher if 0-1 prior orders)
    recency_base = local_rng.gamma(shape=2.5, scale=12.0, size=n_customers)
    recency_penalty = (previous_orders <= 1) * 15.0
    days_since_last_order = np.clip(recency_base + recency_penalty, 1, 180).round().astype(int)
    
    # Historical average order value in dollars ($12 to $250)
    aov_noise = local_rng.normal(0.0, 10.0, size=n_customers)
    avg_order_value = np.clip(
        45.0 + 1.2 * np.minimum(previous_orders, 25) + aov_noise,
        12.0, 250.0
    ).round(2)
    
    # Historical marketing email engagement score in [0, 1]
    prior_email_engagement = local_rng.beta(a=2.0, b=5.0, size=n_customers).round(4)
    
    # Price sensitivity metric in [0, 1] (Observable psychographic trait)
    price_sensitivity = local_rng.beta(a=3.0, b=3.0, size=n_customers).round(4)
    
    # -------------------------------------------------------------
    # 2. Confounded Treatment Assignment: discount_percent [0.00 to 0.30]
    # -------------------------------------------------------------
    # Retailers target price-sensitive customers, churn risks, and email openers,
    # while withholding discounts from high-spenders who buy at full price.
    recency_std = (days_since_last_order - 30.0) / 30.0
    discount_latent_score = (
        -0.80
        + 1.80 * price_sensitivity
        + 0.50 * recency_std
        + 1.20 * prior_email_engagement
        - 0.004 * (avg_order_value - 50.0)
        + 0.020 * (previous_orders - 5.0)
        + local_rng.normal(0.0, 0.40, size=n_customers)
    )
    
    # Zero-hurdle policy: non-positive latent score receives 0% discount
    discount_percent = np.where(
        discount_latent_score <= 0.0,
        0.0,
        np.clip(0.05 + 0.15 * discount_latent_score, 0.05, 0.30)
    ).round(4)
    
    # -------------------------------------------------------------
    # 3. Outcome Generation: purchased (binary 0 or 1)
    # -------------------------------------------------------------
    # Baseline conversion logit (without treatment)
    base_purchase_logit = (
        -2.20
        + 0.0003 * tenure_days
        + 0.080 * previous_orders
        - 0.030 * days_since_last_order
        + 0.015 * avg_order_value
        + 1.60 * prior_email_engagement
        - 0.80 * price_sensitivity  # Price-sensitive shoppers buy less at full price
    )
    
    # Heterogeneous treatment effect: tau_i = (1.5 + 4.5 * price_sensitivity)
    # Discounts are substantially more persuasive for price-sensitive customers
    individual_treatment_effect = (1.5 + 4.5 * price_sensitivity) * discount_percent
    
    purchase_logit = base_purchase_logit + individual_treatment_effect
    purchase_prob = 1.0 / (1.0 + np.exp(-np.clip(purchase_logit, -20.0, 20.0)))
    purchased = local_rng.binomial(1, purchase_prob)
    
    # -------------------------------------------------------------
    # 4. Downstream Metric: revenue ($0 if no purchase, else discounted basket)
    # -------------------------------------------------------------
    basket_variation = local_rng.lognormal(mean=0.0, sigma=0.15, size=n_customers)
    revenue = np.where(
        purchased == 1,
        np.maximum(5.0, avg_order_value * (1.0 - discount_percent) * basket_variation),
        0.0
    ).round(2)
    
    # -------------------------------------------------------------
    # 5. Assemble final observed DataFrame
    # -------------------------------------------------------------
    df = pd.DataFrame({
        "customer_id": customer_id,
        "tenure_days": tenure_days,
        "previous_orders": previous_orders,
        "days_since_last_order": days_since_last_order,
        "avg_order_value": avg_order_value,
        "prior_email_engagement": prior_email_engagement,
        "price_sensitivity": price_sensitivity,
        "discount_percent": discount_percent,
        "purchased": purchased,
        "revenue": revenue,
    })
    
    return df

# Generate the Week 1 dataset
retail_df = generate_synthetic_retail_data(n_customers=5000, seed=RANDOM_SEED)

print(f"Generated retail dataset: {len(retail_df):,} rows x {retail_df.shape[1]} columns")
display(retail_df.head(10))


Generated retail dataset: 5,000 rows x 10 columns


,customer_id,tenure_days,previous_orders,days_since_last_order,avg_order_value,prior_email_engagement,price_sensitivity,discount_percent,purchased,revenue
0,CUST_00001,161,0,32,31.82,0.4174,0.6310,0.0898,0,0.00
1,CUST_00002,1167,17,13,54.32,0.1633,0.4697,0.1360,0,0.00
2,CUST_00003,992,17,22,62.41,0.4521,0.2651,0.1195,1,39.79
3,CUST_00004,675,5,29,58.61,0.0635,0.3715,0.0000,0,0.00
4,CUST_00005,666,5,30,48.45,0.2264,0.5542,0.1763,0,0.00
5,CUST_00006,1292,15,40,59.68,0.4800,0.3241,0.2803,0,0.00
6,CUST_00007,156,2,62,51.37,0.3206,0.5184,0.2042,0,0.00
7,CUST_00008,1055,15,27,63.28,0.3705,0.5577,0.0789,1,61.17
8,CUST_00009,326,3,36,44.19,0.4520,0.6018,0.1683,0,0.00
9,CUST_00010,168,4,6,39.78,0.3336,0.8105,0.0000,0,0.00


## 4. Feature Dictionary and Causal Taxonomy

In causal inference, features cannot simply be tossed into a model as generic predictive covariates. Every feature must be classified by its **temporal status** (pre-treatment vs. post-treatment) and its **structural causal role**.

### 4.1 Feature Documentation Table

| Column Name | Data Type | Business Meaning | Generating Distribution | Temporal Timing | Causal Role | Why It Matters for Causal Inference |
| :--- | :--- | :--- | :--- | :--- | :--- | :--- |
| `customer_id` | String | Unique customer identifier (`CUST_00001`...) | Sequential index | Exogenous | **Identifier** | Non-causal entity index. Must be excluded from causal graphs and estimators to avoid spurious memorization. |
| `tenure_days` | Integer | Account age with the retailer in days | $\text{Uniform}(30, 1500)$ | Pre-treatment | **Confounder** | Older accounts have higher loyalty and distinct responsiveness; affects prior orders, targeting eligibility, and baseline conversion. |
| `previous_orders` | Integer | Cumulative order count prior to intervention | $\text{Poisson}(\lambda(\text{tenure}))$ | Pre-treatment | **Confounder** | Measures purchasing frequency. Influences recency, historical order value, discount assignment, and baseline purchase intent. |
| `days_since_last_order` | Integer | Days elapsed since most recent purchase | $\text{Gamma}(2.5, 12) + \text{penalty}$ | Pre-treatment | **Confounder** | Measures churn risk. Retailers aggressively target inactive customers with win-back discounts; recency strongly depresses baseline conversion. |
| `avg_order_value` | Float | Historical average order value in dollars | Shifted Normal + $\text{orders}$ | Pre-treatment | **Confounder** | Spending capacity. Retailers avoid discounting high-AOV customers to protect margins; AOV also directly scales downstream revenue. |
| `prior_email_engagement` | Float | Historical marketing email click/open rate $[0, 1]$ | $\text{Beta}(2, 5)$ | Pre-treatment | **Confounder** | Marketing receptivity. Engaged subscribers are more likely to receive promotional emails and also convert at higher baseline rates. |
| `price_sensitivity` | Float | Simulated customer sensitivity to price $[0, 1]$ | $\text{Beta}(3, 3)$ | Pre-treatment | **Confounder & Effect Modifier** | Deal responsiveness. Pricing algorithms target price-sensitive shoppers; also interacts with discount to govern treatment effect size (CATE). |
| `discount_percent` | Float | Continuous discount percentage offered $[0.0, 0.30]$ | Confounded Hurdle Model | **Intervention** | **Treatment ($T$)** | The actionable policy variable of interest. We seek to evaluate the causal effect of setting this variable via an external intervention $do(T=t)$. |
| `purchased` | Integer | Binary purchase indicator during follow-up ($0/1$) | Bernoulli via logistic link | Post-treatment | **Primary Outcome ($Y$)** | Primary conversion metric. Jointly determined by customer baseline attributes and the assigned discount treatment. |
| `revenue` | Float | Simulated follow-up revenue in dollars | Zero if no purchase; basket value | Post-treatment | **Downstream Metric** | Commercial business objective. Post-treatment consequence of `purchased` and `discount_percent`. Conditioning on revenue would induce collider/mediator bias! |

---

### 4.2 Distinguishing Confounders from Predictive Features
A critical principle of causal machine learning is:
> **A feature is NOT a confounder merely because it predicts the outcome.**

A variable $W$ is a **confounder** if and only if it is a common cause of **both** the treatment $T$ and the outcome $Y$ ($W \rightarrow T$ and $W \rightarrow Y$), thereby opening a non-causal **backdoor path** between them:

$$T \leftarrow W \rightarrow Y$$

- Pre-treatment variables like `days_since_last_order` and `price_sensitivity` are genuine confounders because they drive both retailer discount assignment and organic customer conversion.
- Post-treatment variables like `revenue` must **never** be conditioned upon as confounders because they occur *after* the treatment. Conditioning on a downstream mediator or collider introduces severe selection bias and washes out the causal effect.


In [3]:
print("=== Summary Statistics ===")
display(retail_df.describe().T)

=== Summary Statistics ===


,count,mean,std,min,25%,50%,75%,max
tenure_days,5000.0,762.5132,425.4665,30.0000,392.7500,756.0000,1135.0000,1499.0000
previous_orders,5000.0,8.3928,5.4097,0.0000,4.0000,8.0000,12.0000,28.0000
days_since_last_order,5000.0,31.4780,19.8391,1.0000,17.0000,27.0000,42.0000,163.0000
avg_order_value,5000.0,54.9047,11.8839,12.0000,46.6725,54.8750,62.9900,94.0100
prior_email_engagement,5000.0,0.2872,0.1568,0.0022,0.1637,0.2692,0.3889,0.9068
price_sensitivity,5000.0,0.5007,0.1918,0.0296,0.3582,0.4978,0.6411,0.9886
discount_percent,5000.0,0.1260,0.0907,0.0000,0.0609,0.1253,0.1910,0.3000
purchased,5000.0,0.3224,0.4674,0.0000,0.0000,0.0000,1.0000,1.0000
revenue,5000.0,16.5074,25.2045,0.0000,0.0000,0.0000,40.2950,99.9500
